# Controller SFT sweep: loss analysis

This notebook analyzes the 8 controller-training runs produced by `run_scripts/run_controller_sft_variant_sweep.sh`: 4 loss/label variants across global test traces and coarse test traces replayed under global context.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_ROOT = PROJECT_ROOT / "outputs" / "controller_sft_m3cot_test_sweeps"
TRAIN_ROOT = RUN_ROOT / "train"
MANIFEST_PATH = RUN_ROOT / "controller_runs.jsonl"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ROOT:", RUN_ROOT)

In [ ]:
def read_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

def load_manifest(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.DataFrame(read_jsonl(path))
    rows = []
    for loss_path in TRAIN_ROOT.glob("*/*/controller_sft_losses.jsonl"):
        rows.append({
            "trace_source": loss_path.parents[1].name,
            "variant": loss_path.parent.name,
            "output_dir": str(loss_path.parent),
            "loss_history_path": str(loss_path),
            "summary_path": str(loss_path.parent / "controller_sft_summary.json"),
            "controller_checkpoint_path": str(loss_path.parent / "controller_sft.pt"),
            "config_path": str(PROJECT_ROOT / "configs" / "qwen2vl_m3cot.yaml"),
        })
    return pd.DataFrame(rows)

manifest = load_manifest(MANIFEST_PATH)
manifest

In [ ]:
loss_rows = []
summary_rows = []

for run in manifest.to_dict("records"):
    loss_path = Path(run["loss_history_path"])
    if loss_path.exists():
        for row in read_jsonl(loss_path):
            row.update({"trace_source": run["trace_source"], "variant": run["variant"]})
            loss_rows.append(row)

    summary_path = Path(run["summary_path"])
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as handle:
            summary = json.load(handle)
        summary_rows.append({
            "trace_source": run["trace_source"],
            "variant": run["variant"],
            "mean_loss": summary.get("mean_loss"),
            "trained_examples": summary.get("trained_examples"),
            "phase3_v2": summary.get("phase3_v2"),
            "use_type_loss_weights": summary.get("use_type_loss_weights"),
            "multi_hot_patch_labels": summary.get("multi_hot_patch_labels"),
            "multi_hot_patch_target_mode": summary.get("multi_hot_patch_target_mode"),
            "multi_hot_patch_blocks": (summary.get("multi_hot_totals") or {}).get("patch_blocks", 0),
            "multi_hot_patch_indices": (summary.get("multi_hot_totals") or {}).get("patch_indices", 0),
            "action_counts": summary.get("action_counts", {}),
            "loss_components": summary.get("loss_components", {}),
            "action_loss_means": summary.get("action_loss_means", {}),
        })

loss_df = pd.DataFrame(loss_rows)
summary_df = pd.DataFrame(summary_rows)

print("loss rows:", len(loss_df))
display(summary_df.sort_values(["trace_source", "variant"]))

In [ ]:
if not loss_df.empty:
    metric_cols = ["total_loss", "action_loss", "region_loss", "patch_loss"]
    rolling_window = 50
    plot_df = loss_df.sort_values(["trace_source", "variant", "global_step"]).copy()
    for metric in metric_cols:
        plot_df[f"{metric}_rolling"] = (
            plot_df.groupby(["trace_source", "variant"])[metric]
            .transform(lambda s: s.rolling(rolling_window, min_periods=1).mean())
        )

    for metric in metric_cols:
        fig, axes = plt.subplots(1, max(1, plot_df["trace_source"].nunique()), figsize=(7 * max(1, plot_df["trace_source"].nunique()), 4), squeeze=False)
        for ax, (trace_source, sub) in zip(axes[0], plot_df.groupby("trace_source")):
            for variant, run_df in sub.groupby("variant"):
                ax.plot(run_df["global_step"], run_df[f"{metric}_rolling"], label=variant)
            ax.set_title(f"{trace_source}: {metric} ({rolling_window}-step rolling mean)")
            ax.set_xlabel("global step")
            ax.set_ylabel(metric)
            ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
if not loss_df.empty:
    final_losses = (
        loss_df.sort_values("global_step")
        .groupby(["trace_source", "variant"])
        .tail(200)
        .groupby(["trace_source", "variant"])[["total_loss", "action_loss", "region_loss", "patch_loss"]]
        .mean()
        .rename(columns=lambda c: "tail200_" + c)
        .reset_index()
    )
    display(final_losses.sort_values(["trace_source", "tail200_total_loss"]))

In [ ]:
component_rows = []
for row in summary_rows:
    base = {"trace_source": row["trace_source"], "variant": row["variant"]}
    for name, value in (row.get("loss_components") or {}).items():
        component_rows.append({**base, "component": name, "loss": value})

component_df = pd.DataFrame(component_rows)
if not component_df.empty:
    display(component_df.pivot_table(index=["trace_source", "variant"], columns="component", values="loss"))
    component_df.pivot_table(index="variant", columns="component", values="loss", aggfunc="mean").plot(kind="bar", figsize=(10, 4))
    plt.title("Mean loss components across trace sources")
    plt.ylabel("loss")
    plt.tight_layout()
    plt.show()

## What to look for

- `normal` is the baseline: no v2 transform, no action-type weights, no multi-hot patch labels.
- `weighted` isolates the effect of action-type weighting.
- `multihot_binary` keeps one controller decision for a patch sequence and supervises all selected patch indices with a binary target.
- `multihot_ordered` uses a rank-decayed soft patch target, so earlier oracle patches get more probability mass.